In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("chrisfilo/fruit-recognition")

print("Path to dataset files:", path)

c:\Users\josej\Documents\pythonCode\TripleTen\kagle\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Path to dataset files: C:\Users\josej\.cache\kagglehub\datasets\chrisfilo\fruit-recognition\versions\1


Import requested libraries 

In [2]:
import os
from glob import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

Data frame creation

In [3]:
classes = sorted([d for d in os.listdir(path)
                  if os.path.isdir(os.path.join(path, d))])

In [ ]:
image_paths = []
for ext in ("jpg", "jpeg", "png", "JPG", "JPEG", "PNG"):
    image_paths.extend(glob(os.path.join(path, "*", f"*.{ext}")))

Valuable functions

In [ ]:
#Importamos las librerias necesarias para el proyecto
import PIL
from PIL import Image
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.layers import Conv2D, Flatten, Dense, AvgPool2D, MaxPooling2D
from tensorflow.keras.models import Sequential
from tensorflow.keras.optimizers import Adam
import numpy as np

#Creacion de funciones para cargar datos, crear modelo y entrenar modelo


def load_data(path):
    datagen = ImageDataGenerator(
        rescale=1.0 / 255.0,
        vertical_flip=True,
        horizontal_flip=True
    )

    data_gen_flow = datagen.flow_from_directory(
        path,
        target_size=(150, 150),
        batch_size=16,
        class_mode='sparse',
        seed=12345
    )

    return data_gen_flow


def create_model(input_shape):
    model = Sequential()

    model.add(Conv2D(6, (5, 5), padding='same', activation='relu',
                     input_shape=input_shape))
                     
    # Añade una capa pooling aquí (MaxPool2D o AvgPool2D)
    model.add(MaxPooling2D(pool_size=(2, 2)))

    model.add(Conv2D(16, (5, 5), padding='valid', activation='relu'))
    
    # Añade una capa pooling aquí (MaxPool2D o AvgPool2D)
    model.add(MaxPooling2D(pool_size=(2, 2)))

    model.add(Conv2D(16, (5, 5), padding='valid', activation='relu'))
    
    # Añade una capa pooling aquí (MaxPool2D o AvgPool2D)
    model.add(MaxPooling2D(pool_size=(2, 2)))

    # Añade una capa flatten aquí
    model.add(Flatten())
    
    # Añade 3 capas dense aquí 
    model.add(Dense(64, activation='relu'))
    model.add(Dense(64, activation='relu'))
    model.add(Dense(12, activation='softmax'))

    optimizer = Adam(learning_rate=0.0001)
    
    model.compile(optimizer=optimizer, 
                                loss= 'sparse_categorical_crossentropy',
                  metrics=['acc'])

    return model



def train_model(model, train_data, test_data, 
                batch_size=None, 
                epochs=10, 
                steps_per_epoch=None, 
                validation_steps=None):

   model.fit(train_data,
          validation_data=test_data,
          batch_size=batch_size,
          epochs=epochs,
          verbose=2)

   return model

input_shape = (150, 150, 3)

train = load_data(path)
test = load_data(path)
model = create_model(input_shape)
model = train_model(model, train, test)


Found 70549 images belonging to 15 classes.
Found 70549 images belonging to 15 classes.


ImportError: Could not import PIL.Image. The use of `load_img` requires PIL.

In [2]:
from tensorflow.keras.applications.resnet import ResNet50

model = ResNet50(input_shape=None,
                 classes=1000,
                 include_top=True,
                 weights='imagenet')

102967424/102967424 ━━━━━━━━━━━━━━━━━━━━ 83s 1us/step


In [3]:
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense
from tensorflow.keras.models import Sequential

backbone = ResNet50(
    input_shape=(150, 150, 3), 
    weights='imagenet', 
    include_top=False
)

model = Sequential()
model.add(backbone)
model.add(GlobalAveragePooling2D())
model.add(Dense(12, activation='softmax'))

94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 20s 0us/step


In [4]:
backbone = ResNet50(
    input_shape=(150, 150, 3), weights='imagenet', include_top=False
)

# congela ResNet50 sin la parte superior
backbone.trainable = False

model = Sequential()
model.add(backbone)
model.add(GlobalAveragePooling2D())
model.add(Dense(12, activation='softmax'))

In [7]:
def load_data(path, width, height, batch_size):
    train_datagen = ImageDataGenerator(
        rescale=1.0 / 255.0,
        vertical_flip=True,
        horizontal_flip=True)
    
    train_generator = train_datagen.flow_from_directory(
        path,
        target_size=(width, height),
        batch_size=batch_size,
        shuffle=True,
        class_mode='sparse',
        seed=12345
    )
    return train_generator

def create_resnet_model(input_shape):
    backbone = ResNet50(
        input_shape=input_shape, weights='imagenet', include_top=False
    )
    backbone.trainable = False

    model = Sequential()
    model.add(backbone)
    model.add(GlobalAveragePooling2D())
    model.add(Dense(12, activation='softmax'))

    optimizer = Adam(learning_rate=0.0001)
    
    model.compile(optimizer=optimizer, 
                  loss='sparse_categorical_crossentropy',
                  metrics=['acc'])

    return model

def train_model(model, train_data, va_data, epochs=10):
    model.fit(
        train_data,
        validation_data=va_data,
        epochs=epochs,
        verbose=2
    )
    return model



In [ ]:
width = 160
height = 128
batch_size = 16
input_shape = (width, height, 3)


